In [1]:
!pip install --upgrade pip setuptools wheel
!pip install --no-deps seqeval
!pip install accelerate
!pip show seqeval | grep Version
print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 42.6 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=cc81194d5e657d3ccfd011fa87ac7daf9ccef53eb635b3dee82263caf843e850
  Stored in directory: /root/.cache/p

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("GPU not enabled. Set Accelerator to T4.")

CUDA available: True
GPU: Tesla T4
VRAM: 15.64 GB


In [3]:
import os, sys, shutil

INPUT_SRC = "/kaggle/input/datasets/purvanshsahu/drise-code/src"
WORKING_SRC = "/kaggle/working/src"

if os.path.exists(WORKING_SRC):
    shutil.rmtree(WORKING_SRC)
shutil.copytree(INPUT_SRC, WORKING_SRC)
print(f"✅ Copied src to {WORKING_SRC}")

INPUT_CFG = "/kaggle/input/datasets/purvanshsahu/drise-code/configs"
WORKING_CFG = "/kaggle/working/configs"
if os.path.exists(INPUT_CFG):
    if os.path.exists(WORKING_CFG):
        shutil.rmtree(WORKING_CFG)
    shutil.copytree(INPUT_CFG, WORKING_CFG)
    print(f"✅ Copied configs to {WORKING_CFG}")

if WORKING_SRC not in sys.path:
    sys.path.insert(0, WORKING_SRC)
print(f"sys.path[0]: {sys.path[0]}")

✅ Copied src to /kaggle/working/src
✅ Copied configs to /kaggle/working/configs
sys.path[0]: /kaggle/working/src


In [4]:
import sys
from collections import Counter
from pathlib import Path

target = Path("/kaggle/working/src/document_intelligence_engine/multimodal/cord_dataset.py")

content = r'''
"""Dataset loaders for LayoutLMv3 fine-tuning.

Supports two receipt sources:
  - CORD receipts, mapping their semantic categories (from ``valid_line``)
    to the project's 5-class BIO scheme (``O``, ``B-KEY``, ``I-KEY``,
    ``B-VALUE``, ``I-VALUE``). CORD tokens flagged ``is_key`` provide the
    KEY supervision; the rest are VALUE spans.
  - FUNSD forms, whose QUESTION/ANSWER annotation is the canonical source of
    KEY (question) and VALUE (answer) supervision. Including FUNSD teaches the
    model to detect printed field names (keys) that receipts do not annotate.

Both the ``naver-clova-ix/cord-v2`` format (``ground_truth`` JSON) and the
``katanaml/cord`` format (``words``/``bboxes``/``ner_tags``) are supported.
"""

from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Any

import torch
from datasets import Dataset, DatasetDict, load_dataset
from PIL import Image, ImageFont
from torch.utils.data import DataLoader
from transformers import LayoutLMv3Processor

logger = logging.getLogger(__name__)

# Project BIO label scheme consumed by postprocessing.entity_grouping
LABEL_LIST = ["O", "B-KEY", "I-KEY", "B-VALUE", "I-VALUE"]
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID2LABEL = {idx: label for idx, label in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

# CORD categories that represent field *names* (keys). CORD v2 annotates the
# values associated with each category, and also marks a small number of
# key-like tokens; everything else with a value is a VALUE span.
_CORD_KEY_CATEGORIES = frozenset({
    "menu.nm",
    "menu.unitprice",
    "menu.cnt",
    "menu.discountprice",
    "menu.sub_nm",
    "menu.sub_unitprice",
    "menu.sub_cnt",
    "menu.etc",
    "menu.vatyn",
    "sub_total.subtotal_price",
    "sub_total.discount_price",
    "sub_total.service_price",
    "sub_total.othersvc_price",
    "sub_total.tax_price",
    "sub_total.etc",
    "total.total_price",
    "total.total_etc",
    "total.cashprice",
    "total.changeprice",
    "total.creditcardprice",
    "total.emoneyprice",
    "total.menutype_cnt",
    "total.menuqty_cnt",
})

# CORD categories whose words describe line-item *values*. These are used to
# recover per-category semantics even though the BIO label stays B/I-VALUE.
_LINE_ITEM_CATEGORIES = frozenset({
    "menu.nm",
    "menu.unitprice",
    "menu.cnt",
    "menu.discountprice",
    "menu.sub_nm",
    "menu.sub_unitprice",
    "menu.sub_cnt",
})


def _cord_label_to_bio(cord_label: str, is_first_token: bool) -> str:
    """Map a CORD category string to our BIO label.

    CORD labels look like ``menu.nm``, ``total.total_price``, etc. These are
    the *values* printed on the receipt, so every non-O token maps to a VALUE
    span. KEY supervision comes from CORD ``is_key`` tokens and FUNSD (see
    ``_parse_cord_v2_example`` and ``_parse_funsd_example``).
    """
    if cord_label == "O" or cord_label.startswith("O"):
        return "O"
    return "B-VALUE" if is_first_token else "I-VALUE"


def _parse_cord_example(example: dict[str, Any], label_names: list[str]) -> dict[str, Any]:
    """Parse a CORD example into flat token lists.

    Supports both the ``ground_truth`` JSON format (cord-v2) and the
    ``words``/``bboxes``/``ner_tags`` format (katanaml/cord).
    """
    if "ground_truth" in example:
        return _parse_cord_v2_example(example)
    if "ner_tags" in example and "words" in example:
        return _parse_cord_flat_example(example, label_names)
    raise ValueError(
        "Unsupported CORD example: expected 'ground_truth' or 'ner_tags'/'words' columns."
    )


def _parse_cord_v2_example(example: dict[str, Any]) -> dict[str, Any]:
    """Parse a CORD v2 example into flat token lists.

    cord-v2 keeps the OCR tokens with their geometry in ``valid_line``
    (line -> ``words`` -> ``text``/``quad``/``is_key``) together with a
    per-line parse ``category`` (e.g. ``menu.nm``). The ``gt_parse`` block
    only contains field *values* without word boxes, so token supervision
    must come from ``valid_line``. Tokens marked ``is_key`` (printed field
    names such as ``TOTAL``/``CASH``) map to KEY spans; everything else with
    a category maps to a VALUE span.
    """
    gt = json.loads(example["ground_truth"])
    valid_line = gt.get("valid_line", [])

    words: list[str] = []
    boxes: list[list[int]] = []
    labels: list[str] = []

    for line_group in valid_line:
        if not isinstance(line_group, dict):
            continue
        category = line_group.get("category")
        for word_info in line_group.get("words", []):
            text = str(word_info.get("text", "")).strip()
            if not text:
                continue
            quad = word_info.get("quad", {})
            if isinstance(quad, dict):
                x_coords = [int(quad.get(f"x{i}", 0)) for i in range(1, 5)]
                y_coords = [int(quad.get(f"y{i}", 0)) for i in range(1, 5)]
            else:
                x_coords = y_coords = []
            boxes.append([min(x_coords), min(y_coords), max(x_coords), max(y_coords)])
            words.append(text)

            if word_info.get("is_key"):
                is_first = not labels or not labels[-1].endswith("-KEY")
                label = "B-KEY" if is_first else "I-KEY"
            elif not category or category == "O":
                label = "O"
            else:
                is_first = not labels or labels[-1] == "O"
                label = _cord_label_to_bio(str(category), is_first)
            labels.append(label)

    return {"words": words, "boxes": boxes, "bio_labels": labels}


def _parse_cord_flat_example(example: dict[str, Any], label_names: list[str]) -> dict[str, Any]:
    """Parse the katanaml/cord format (``words``, ``bboxes``, ``ner_tags``)."""
    raw_tags = example["ner_tags"]
    words: list[str] = []
    boxes: list[list[int]] = []
    labels: list[str] = []
    for word, bbox, tag_id in zip(example["words"], example["bboxes"], raw_tags, strict=False):
        text = str(word).strip()
        if not text:
            continue
        if not bbox:
            bbox = [0, 0, 0, 0]
        tag = label_names[tag_id] if isinstance(tag_id, int) and tag_id < len(label_names) else str(tag_id)
        words.append(text)
        boxes.append([int(value) for value in bbox])
        is_first = not labels or labels[-1] == "O"
        labels.append(_cord_label_to_bio(tag, is_first))

    return {"words": words, "boxes": boxes, "bio_labels": labels}


def _parse_funsd_example(example: dict[str, Any], label_names: list[str]) -> dict[str, Any]:
    """Parse a FUNSD example into KEY/VALUE BIO labels.

    FUNSD annotates QUESTION spans (field names) and ANSWER spans (values),
    which is exactly the KEY -> VALUE structure the post-processing layer
    expects. QUESTION tokens map to B-KEY/I-KEY, ANSWER tokens to B-VALUE/I-VALUE.
    """
    words: list[str] = []
    boxes: list[list[int]] = []
    labels: list[str] = []
    for word, bbox, tag_id in zip(example["words"], example["bboxes"], example["ner_tags"], strict=False):
        text = str(word).strip()
        if not text:
            continue
        tag = label_names[tag_id] if isinstance(tag_id, int) and tag_id < len(label_names) else str(tag_id)
        prefix, base = _split_bio_prefix(tag)
        if base in ("QUESTION", "ANSWER"):
            bio_base = "KEY" if base == "QUESTION" else "VALUE"
            labels.append(f"{prefix}-{bio_base}")
        else:
            labels.append("O")
        if not bbox:
            bbox = [0, 0, 0, 0]
        words.append(text)
        boxes.append([int(value) for value in bbox])

    return {"words": words, "boxes": boxes, "bio_labels": labels}


def _split_bio_prefix(tag: str) -> tuple[str, str]:
    """Split ``B-QUESTION`` into ``(B, QUESTION)``; ``O`` into ``(O, O)``."""
    if "-" in tag:
        prefix, base = tag.split("-", maxsplit=1)
        if prefix in ("B", "I"):
            return prefix, base
    return "O", tag


def _normalize_bbox(box: list[int], width: int, height: int) -> list[int]:
    """Normalize bounding box to 0-1000 range as expected by LayoutLMv3."""
    if width <= 0 or height <= 0:
        return [0, 0, 0, 0]
    return [
        max(0, min(1000, int(1000 * box[0] / width))),
        max(0, min(1000, int(1000 * box[1] / height))),
        max(0, min(1000, int(1000 * box[2] / width))),
        max(0, min(1000, int(1000 * box[3] / height))),
    ]


class _LayoutLMDataset(torch.utils.data.Dataset):
    """Base dataset that tokenizes words/boxes/labels with a LayoutLMv3 processor."""

    def __init__(
        self,
        examples: list[dict[str, Any]],
        processor: LayoutLMv3Processor,
        max_length: int = 512,
        label_names: list[str] | None = None,
    ) -> None:
        self._examples = examples
        self._processor = processor
        self._max_length = max_length
        self._label_names = list(label_names or [])

    def __len__(self) -> int:
        return len(self._examples)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        example = self._examples[idx]
        image = example.get("image")
        if image is None:
            image_path = str(example.get("image_path", ""))
            if image_path and Path(image_path).expanduser().exists():
                image = Image.open(image_path).convert("RGB")
            else:
                image = _render_image_from_tokens(example)
        else:
            image = image.convert("RGB")
        width, height = image.size

        parsed = self._parse(example)
        words = parsed["words"]
        boxes = parsed["boxes"]
        bio_labels = parsed["bio_labels"]

        if not words:
            words = ["[EMPTY]"]
            boxes = [[0, 0, 0, 0]]
            bio_labels = ["O"]

        normalized_boxes = [_normalize_bbox(b, width, height) for b in boxes]

        encoding = self._processor(
            image,
            words,
            boxes=normalized_boxes,
            max_length=self._max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = self._align_labels(encoding, bio_labels)

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "bbox": encoding["bbox"].squeeze(0),
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "labels": labels,
        }

    def _parse(self, example: dict[str, Any]) -> dict[str, Any]:
        raise NotImplementedError

    def _align_labels(
        self,
        encoding: dict[str, torch.Tensor],
        bio_labels: list[str],
    ) -> torch.Tensor:
        word_ids = encoding.word_ids(batch_index=0)
        aligned = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != previous_word_id:
                if word_id < len(bio_labels):
                    aligned.append(LABEL2ID[bio_labels[word_id]])
                else:
                    aligned.append(-100)
            else:
                if word_id < len(bio_labels):
                    label = bio_labels[word_id]
                    if label.startswith("B-"):
                        label = "I-" + label[2:]
                    aligned.append(LABEL2ID.get(label, LABEL2ID["O"]))
                else:
                    aligned.append(-100)
            previous_word_id = word_id

        return torch.tensor(aligned, dtype=torch.long)


class CORDDataset(_LayoutLMDataset):
    def _parse(self, example: dict[str, Any]) -> dict[str, Any]:
        return _parse_cord_example(example, self._label_names)


class FUNSDDataset(_LayoutLMDataset):
    def _parse(self, example: dict[str, Any]) -> dict[str, Any]:
        return _parse_funsd_example(example, self._label_names)


def _feature_label_names(dataset: Dataset) -> list[str]:
    feature = dataset.features.get("ner_tags")
    if feature is None:
        return []
    while hasattr(feature, "feature"):
        feature = feature.feature
    if hasattr(feature, "names"):
        return list(feature.names)
    return []


def _render_image_from_tokens(example: dict[str, Any]) -> Image.Image:
    """Render a white-canvas image from words + bounding boxes.

    Used when a dataset stores ``image_path`` references that are not
    available locally (e.g. the katanaml/cord cache on another machine).
    """
    from PIL import ImageDraw

    canvas_size = 1200
    image = Image.new("RGB", (canvas_size, canvas_size), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    words = example.get("words", [])
    bboxes = example.get("bboxes", [])
    for word, bbox in zip(words, bboxes, strict=False):
        text = str(word).strip()
        if not text or not bbox:
            continue
        x0, y0, x1, y1 = [max(0, min(canvas_size - 1, int(value * 1.1))) for value in bbox]
        if x1 <= x0:
            x1 = min(canvas_size - 1, x0 + 40)
        if y1 <= y0:
            y1 = min(canvas_size - 1, y0 + 20)
        draw.rectangle([x0, y0, x1, y1], outline="lightgray", width=1)
        draw.text((x0 + 2, y0 + 1), text, fill="black", font=font)
    return image


def load_cord_dataset(max_train_samples: int | None = None) -> DatasetDict:
    """Load a CORD dataset, preferring cord-v2 and falling back to katanaml/cord."""
    dataset: DatasetDict | None = None
    try:
        dataset = load_dataset("naver-clova-ix/cord-v2")
    except Exception as exc:  # pragma: no cover - network dependent
        logger.warning("Failed to load naver-clova-ix/cord-v2 (%s); trying katanaml/cord", exc)
        dataset = load_dataset("katanaml/cord")

    if max_train_samples is not None and max_train_samples > 0:
        train_size = min(max_train_samples, len(dataset["train"]))
        dataset["train"] = dataset["train"].select(range(train_size))
        logger.info("Limited training set to %d samples", train_size)

    logger.info(
        "CORD dataset loaded: train=%d, validation=%d, test=%d",
        len(dataset["train"]),
        len(dataset["validation"]),
        len(dataset["test"]),
    )
    return dataset


def get_cord_dataloaders(
    model_name: str = "microsoft/layoutlmv3-base",
    batch_size: int = 4,
    max_length: int = 512,
    max_train_samples: int | None = None,
    include_funsd: bool = False,
) -> tuple[DataLoader, DataLoader, list[str]]:
    """Build train and validation DataLoaders for CORD (+ optional FUNSD).

    Returns:
        (train_loader, val_loader, label_list)
    """
    processor = LayoutLMv3Processor.from_pretrained(
        model_name,
        apply_ocr=False,  # We supply our own OCR tokens
    )

    raw_dataset = load_cord_dataset(max_train_samples=max_train_samples)

    cord_train_labels = _feature_label_names(raw_dataset["train"])
    train_dataset = CORDDataset(
        [dict(record) for record in raw_dataset["train"]],
        processor,
        max_length=max_length,
        label_names=cord_train_labels,
    )
    val_dataset = CORDDataset(
        [dict(record) for record in raw_dataset["validation"]],
        processor,
        max_length=max_length,
        label_names=_feature_label_names(raw_dataset["validation"]) or cord_train_labels,
    )

    if include_funsd:
        try:
            funsd = load_dataset("nielsr/funsd")
        except Exception as exc:  # pragma: no cover - network dependent
            logger.warning("FUNSD unavailable (%s); continuing with CORD only", exc)
            funsd = None

        if funsd is not None:
            funsd_train_labels = _feature_label_names(funsd["train"])
            funsd_train = FUNSDDataset(
                [dict(record) for record in funsd["train"]],
                processor,
                max_length=max_length,
                label_names=funsd_train_labels,
            )
            # nielsr/funsd has no "validation" split; fall back to "test".
            funsd_val_split = funsd.get("validation", funsd["test"])
            funsd_val = FUNSDDataset(
                [dict(record) for record in funsd_val_split],
                processor,
                max_length=max_length,
                label_names=funsd_train_labels,
            )
            train_dataset = torch.utils.data.ConcatDataset([train_dataset, funsd_train])
            val_dataset = torch.utils.data.ConcatDataset([val_dataset, funsd_val])
            logger.info(
                "Included FUNSD: train += %d, val += %d",
                len(funsd["train"]),
                len(funsd_val_split),
            )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    return train_loader, val_loader, LABEL_LIST
'''

target.write_text(content, encoding="utf-8")
print(f"Wrote {len(content)} chars to {target}")

# Fresh import from the on-disk copy that the training subprocess will read
for mod in list(sys.modules):
    if "document_intelligence_engine.multimodal.cord_dataset" in mod:
        del sys.modules[mod]

from datasets import load_dataset
from document_intelligence_engine.multimodal.cord_dataset import _parse_cord_v2_example

ds = load_dataset("naver-clova-ix/cord-v2", split="train")
cnt = Counter()
for i in range(20):
    cnt.update(_parse_cord_v2_example(ds[i])["bio_labels"])
print("Label histogram (first 20 samples):", dict(cnt))

assert sum(n for lbl, n in cnt.items() if lbl != "O") > 0, "STILL ALL-O -> do NOT train"
print("PARSER OK: real KEY/VALUE labels present - safe to run training")

Wrote 17130 chars to /kaggle/working/src/document_intelligence_engine/multimodal/cord_dataset.py


README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00004-7dbbe248962764(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

...

data/test-00000-of-00001-9c204eb3f4e1179(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Label histogram (first 20 samples): {'B-VALUE': 20, 'I-VALUE': 500, 'B-KEY': 91, 'I-KEY': 33}
PARSER OK: real KEY/VALUE labels present - safe to run training


In [5]:
print("🚀 Starting training with CLI flags...")

!cd /kaggle/working && PYTHONPATH=/kaggle/working/src \
    python -m document_intelligence_engine.multimodal.training \
    --include-funsd \
    --num-epochs 15 \
    --batch-size 4 \
    --gradient-accumulation-steps 2 \
    --learning-rate 5e-5

print("\n✅ Training completed!")

🚀 Starting training with CLI flags...
2026-08-14T10:10:57 INFO __main__ === LayoutLMv3 Training on CORD ===
2026-08-14T10:10:57 INFO __main__ Device: cuda
2026-08-14T10:10:57 INFO __main__ Epochs: 15, LR: 5e-05, Batch: 4, Accum: 2
2026-08-14T10:10:57 INFO __main__ Loading CORD dataset...
2026-08-14T10:10:57 INFO httpx HTTP Request: GET https://huggingface.co/api/models/microsoft/layoutlmv3-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-14T10:10:57 INFO httpx HTTP Request: HEAD https://huggingface.co/microsoft/layoutlmv3-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-08-14T10:10:57 WARNING huggingface_hub.utils._http Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-14T10:10:57 INFO httpx HTTP Request: HEAD https://huggingface.co/microsoft/layoutlmv3-base/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
202

In [6]:
import os, shutil

base_dir = "/kaggle/working/experiments/artifacts/cord_finetuned"
final_dir = os.path.join(base_dir, "final")

best_ckpt = None
for root, dirs, files in os.walk(base_dir):
    if 'pytorch_model.bin' in files or 'model.safetensors' in files:
        best_ckpt = root
        break

if best_ckpt:
    print(f"✅ Checkpoint found: {best_ckpt}")
    os.makedirs(final_dir, exist_ok=True)
    for item in os.listdir(best_ckpt):
        src = os.path.join(best_ckpt, item)
        dst = os.path.join(final_dir, item)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
        elif os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    
    zip_path = "/kaggle/working/cord_finetuned_checkpoint"
    shutil.make_archive(zip_path, 'zip', final_dir)
    zip_size = os.path.getsize(zip_path + ".zip") / 1e6
    print(f"\n✅ Zipped: {zip_path}.zip ({zip_size:.1f} MB)")
    print("📥 Download from Output tab.")
else:
    print("❌ No checkpoint found. Check training logs.")

✅ Checkpoint found: /kaggle/working/experiments/artifacts/cord_finetuned/best

✅ Zipped: /kaggle/working/cord_finetuned_checkpoint.zip (466.3 MB)
📥 Download from Output tab.
